# Bài thực hành 3 — Phân loại ảnh sử dụng mạng tích chập

MSSV: 24521185
Họ và tên: Nguyễn Anh Nguyên

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split

from tensorflow.keras import layers, models
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.optimizers import Adam, SGD, Lion
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping

## Đọc dữ liệu

In [ ]:
def load_images_from_folder(folder, img_rows, img_cols):
    images, labels = [], []
    class_names = sorted(name for name in os.listdir(folder) if os.path.isdir(os.path.join(folder, name)))
    for label, class_name in enumerate(class_names):
        class_folder = os.path.join(folder, class_name)
        for filename in sorted(os.listdir(class_folder)):
            if not filename.lower().endswith('.jpg'):
                continue
            img_path = os.path.join(class_folder, filename)
            img = cv2.imread(img_path)
            if img is None:
                img = cv2.cvtColor(np.array(Image.open(img_path).convert('RGB')), cv2.COLOR_RGB2BGR)
            img = cv2.resize(img, (img_rows, img_cols))
            images.append(img)
            labels.append(label)
    return images, labels, class_names

In [ ]:
train_folder = '/content/drive/MyDrive/CS231/Dataset/HoaVietNam2027/train'
test_folder = '/content/drive/MyDrive/CS231/Dataset/HoaVietNam2027/test'

img_rows, img_cols = 128, 128

training_images, training_labels, class_names = load_images_from_folder(train_folder, img_rows, img_cols)
test_images, test_labels, _ = load_images_from_folder(test_folder, img_rows, img_cols)
num_classes = len(class_names)

## Chia tập train thành train và val theo tỉ lệ 8:2

In [ ]:
train_images, validation_images, train_labels, validation_labels = train_test_split(
    training_images, training_labels, test_size=0.2, train_size=0.8, random_state=42)

## Xử lý dữ liệu

In [ ]:
train_images = np.array(train_images).astype('float32') / 255
validation_images = np.array(validation_images).astype('float32') / 255
test_images = np.array(test_images).astype('float32') / 255

train_labels = to_categorical(train_labels, num_classes)
validation_labels = to_categorical(validation_labels, num_classes)
test_labels = to_categorical(test_labels, num_classes)

print(class_names)
print('Train:', train_images.shape, train_labels.shape)
print('Val:', validation_images.shape, validation_labels.shape)
print('Test:', test_images.shape, test_labels.shape)

## Mạng CNN dựa trên LeNet-5

Tập HoaVietNam2027 có 8 lớp nên lớp Dense cuối dùng 8 nơ-ron.

In [ ]:
def build_model():
    model = models.Sequential()
    model.add(layers.Conv2D(6, (3, 3), activation='relu', input_shape=(img_rows, img_cols, 3), padding='same'))
    model.add(layers.MaxPooling2D((2, 2)))
    model.add(layers.Conv2D(16, (3, 3), activation='relu', padding='same'))
    model.add(layers.MaxPooling2D((2, 2)))
    model.add(layers.Flatten())
    model.add(layers.Dense(120, activation='relu'))
    model.add(layers.Dense(84, activation='relu'))
    model.add(layers.Dense(num_classes, activation='softmax'))
    return model

build_model().summary()

# Yêu cầu 1
Thực nghiệm với 3 optimizer Adam, SGD, Lion; learning_rate=0.00001, loss='categorical_crossentropy', metrics='accuracy', batch_size=8, epochs=15. Đánh giá 3 mô hình trên tập test.

In [ ]:
optimizers = {
    'Adam': Adam,
    'SGD': SGD,
    'Lion': Lion,
}
results = {}

for name, optimizer in optimizers.items():
    print(f'===== {name} =====')
    model = build_model()
    model.compile(optimizer=optimizer(learning_rate=0.00001), loss='categorical_crossentropy', metrics=['accuracy'])
    model.fit(train_images, train_labels, epochs=15, batch_size=8,
              validation_data=(validation_images, validation_labels))

    test_loss, test_accuracy = model.evaluate(test_images, test_labels, verbose=2)
    predicted_labels = np.argmax(model.predict(test_images), axis=-1)
    results[name] = (test_loss, test_accuracy)
    print(f'Test Accuracy: {test_accuracy}')
    print(f'Test Loss: {test_loss}')
    print(classification_report(np.argmax(test_labels, axis=-1), predicted_labels,
                                target_names=class_names, zero_division=0))

In [ ]:
for name, (test_loss, test_accuracy) in results.items():
    print(f'{name:<5} Test Accuracy = {test_accuracy:.4f}  Test Loss = {test_loss:.4f}')

best_optimizer = max(results, key=lambda name: results[name][1])
print('Optimizer phù hợp nhất:', best_optimizer)

# Yêu cầu 2
Huấn luyện lại với optimizer tốt nhất ở yêu cầu 1, có sử dụng ReduceLROnPlateau và EarlyStopping. Các tham số khác giữ như yêu cầu 1.

In [ ]:
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-7, verbose=1)
early_stopping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=1)

model = build_model()
model.compile(optimizer=optimizers[best_optimizer](learning_rate=0.00001),
              loss='categorical_crossentropy', metrics=['accuracy'])
history = model.fit(train_images, train_labels, epochs=15, batch_size=8,
                    validation_data=(validation_images, validation_labels),
                    callbacks=[reduce_lr, early_stopping])

## Biểu đồ độ chính xác và loss trong quá trình huấn luyện

In [ ]:
plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'])
plt.title(f'model accuracy ({best_optimizer})')
plt.ylabel('accuracy')
plt.xlabel('epoch')
plt.legend(['train', 'val'], loc='upper left')
plt.show()

plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])
plt.title(f'model loss ({best_optimizer})')
plt.ylabel('loss')
plt.xlabel('epoch')
plt.legend(['train', 'val'], loc='upper left')
plt.show()

## Đánh giá trên tập test

In [ ]:
test_loss, test_accuracy = model.evaluate(test_images, test_labels, verbose=2)
print(f'Test Accuracy: {test_accuracy}')
print(f'Test Loss: {test_loss}')

predicted_labels = np.argmax(model.predict(test_images), axis=-1)
print(classification_report(np.argmax(test_labels, axis=-1), predicted_labels,
                            target_names=class_names, zero_division=0))